In [2]:
import pandas as pd

# Charger le CSV
df = pd.read_csv("chicago_crimes_2015_2025.csv")

print(df.shape)
df.head()

(2747492, 22)


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,14061904,JJ530178,12/20/2025 12:00:00 AM,005XX W CONGRESS PKWY,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,False,False,...,28.0,28.0,07,1173014.0,1897914.0,2025,12/27/2025 03:44:33 PM,"41,875306962","-87,640223321","(41.875306962, -87.640223321)"
1,14063999,JJ532646,12/20/2025 12:00:00 AM,043XX N HAZEL ST,0820,THEFT,$500 AND UNDER,RESIDENCE,False,False,...,46.0,3.0,06,1169474.0,1929172.0,2025,12/27/2025 03:44:33 PM,"41,961158296","-87,652309344","(41.961158296, -87.652309344)"
2,14064542,JJ533236,12/20/2025 12:00:00 AM,050XX S WASHINGTON PARK CT,0820,THEFT,$500 AND UNDER,APARTMENT,False,False,...,20.0,38.0,06,1180109.0,1871497.0,2025,12/27/2025 03:44:33 PM,"41,802656712","-87,614984788","(41.802656712, -87.614984788)"
3,14061781,JJ529904,12/20/2025 12:00:00 AM,057XX S HOMAN AVE,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,False,True,...,14.0,63.0,08B,1154693.0,1866369.0,2025,12/27/2025 03:44:33 PM,"41,789129824","-87,708333583","(41.789129824, -87.708333583)"
4,14063605,JJ532185,12/20/2025 12:00:00 AM,001XX S ST LOUIS AVE,0820,THEFT,$500 AND UNDER,STREET,False,False,...,28.0,27.0,06,1153034.0,1899168.0,2025,12/27/2025 03:44:33 PM,"41,879167386","-87,713548922","(41.879167386, -87.713548922)"


In [3]:
# Uniformiser les noms de colonnes
df.columns = df.columns.str.lower().str.replace(" ", "_")

# Conversion date
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Supprimer lignes sans date ou zone
df = df.dropna(subset=['date', 'community_area'])

# Supprimer doublons
df = df.drop_duplicates(subset=['id'])

print(df.shape)


C:\Users\2020\AppData\Local\Temp\ipykernel_12484\245422602.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')


(2747314, 22)


In [4]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter


In [5]:
df['arrest'] = df['arrest'].astype(int)
df['domestic'] = df['domestic'].astype(int)


In [6]:
import numpy as np

def classify_gravity(primary_type):
    if primary_type in [
        "HOMICIDE",
        "CRIMINAL SEXUAL ASSAULT",
        "HUMAN TRAFFICKING",
        "KIDNAPPING",
        "ARSON",
        "ROBBERY",
        "MOTOR VEHICLE THEFT",
        "OFFENSE INVOLVING CHILDREN",
        "SEX OFFENSE",
        "STALKING"
    ]:
        return "Très grave"

    elif primary_type in [
        "ASSAULT",
        "BATTERY",
        "INTIMIDATION",
        "INTERFERENCE WITH PUBLIC OFFICER",
        "WEAPONS VIOLATION",
        "BURGLARY",
        "THEFT",
        "CRIMINAL DAMAGE",
        "OTHER NARCOTIC VIOLATION",
        "NARCOTICS"
    ]:
        return "Grave"

    elif primary_type in [
        "CRIMINAL TRESPASS",
        "DECEPTIVE PRACTICE",
        "GAMBLING",
        "LIQUOR LAW VIOLATION",
        "CONCEALED CARRY LICENSE VIOLATION",
        "PUBLIC PEACE VIOLATION"
    ]:
        return "Modéré"

    elif primary_type in [
        "PUBLIC INDECENCY",
        "OBSCENITY",
        "PROSTITUTION",
        "NON-CRIMINAL",
        "OTHER OFFENSE"
    ]:
        return "Mineur"

    else:
        return "Non classé"


df['crime_severity_label'] = df['primary_type'].apply(classify_gravity)


In [7]:
cols_to_drop = [
    'id', 'case_number', 'block', 'description',
    'iucr', 'fbi_code', 'updated_on', 'location'
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])


In [8]:
monthly = (
    df
    .groupby(['year', 'month', 'community_area'])
    .size()
    .reset_index(name='crime_count')
)


In [9]:
monthly = monthly.sort_values(['community_area', 'year', 'month'])

monthly['lag_1'] = monthly.groupby('community_area')['crime_count'].shift(1)
monthly['lag_3'] = monthly.groupby('community_area')['crime_count'].shift(3)

monthly = monthly.dropna()


In [10]:
monthly.head()

,year,month,community_area,crime_count,lag_1,lag_3
231,2015,4,1.0,238,293.0,296.0
308,2015,5,1.0,332,238.0,246.0
385,2015,6,1.0,298,332.0,293.0
462,2015,7,1.0,361,298.0,238.0
539,2015,8,1.0,314,361.0,332.0


In [11]:
monthly.to_csv("crime_monthly_ml_ready.csv", index=False)

In [12]:
X = monthly[['year', 'month', 'community_area', 'lag_1', 'lag_3']]
y = monthly['crime_count']


In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X['community_area'] = le.fit_transform(X['community_area'])


C:\Users\2020\AppData\Local\Temp\ipykernel_12484\3090467131.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['community_area'] = le.fit_transform(X['community_area'])


In [14]:
train = monthly[monthly['year'] < 2025]
test  = monthly[monthly['year'] == 2025]

X_train = train[['year', 'month', 'community_area', 'lag_1', 'lag_3']]
y_train = train['crime_count']

X_test  = test[['year', 'month', 'community_area', 'lag_1', 'lag_3']]
y_test  = test['crime_count']


In [15]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


RandomForestRegressor(max_depth=15, n_estimators=200, n_jobs=-1,
                      random_state=42)

In [16]:
y_pred = model.predict(X_test)


In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


MAE : 29.67269023759888
RMSE: 49.78558678291374
R²  : 0.9422794707305604


In [18]:
import pandas as pd

importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

print(importance)


          feature  importance
3           lag_1    0.973703
4           lag_3    0.010471
1           month    0.007862
2  community_area    0.004654
0            year    0.003309


In [19]:
baseline_pred = X_test['lag_1']

baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("Baseline MAE:", baseline_mae)
print("Model MAE   :", mae)


Baseline MAE: 34.25865800865801
Model MAE   : 29.67269023759888


In [21]:
import xgboost as xgb

In [22]:
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [23]:
y_pred_xgb = xgb_model.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost MAE :", mae_xgb)
print("XGBoost RMSE:", rmse_xgb)
print("XGBoost R²  :", r2_xgb)


XGBoost MAE : 28.451387405395508
XGBoost RMSE: 49.01550794933426
XGBoost R²  : 0.9440512657165527


In [24]:
import pandas as pd

importance_xgb = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values(by='importance', ascending=False)

print(importance_xgb)


          feature  importance
3           lag_1    0.781537
4           lag_3    0.189684
1           month    0.015897
0            year    0.006540
2  community_area    0.006342


In [25]:
import pandas as pd
import numpy as np

# Toutes les zones existantes
zones = df['community_area'].unique()

# Générer toutes les combinaisons mois x zone pour 2026 et 2027
future_months = pd.DataFrame([
    {'year': year, 'month': month, 'community_area': zone}
    for year in [2026, 2027]
    for month in range(1, 13)
    for zone in zones
])


In [26]:
# Derniers mois connus pour chaque zone
last_known = monthly.groupby('community_area').apply(lambda x: x.sort_values(['year','month']).tail(3)).reset_index(drop=True)

# Dictionnaire pour garder la mémoire des lags
lag_dict = {}

for zone in zones:
    # Trier par date
    temp = last_known[last_known['community_area'] == zone].sort_values(['year','month'])
    lag_dict[zone] = list(temp['crime_count'].values)  # [lag3, lag2, lag1]


C:\Users\2020\AppData\Local\Temp\ipykernel_12484\4197661625.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  last_known = monthly.groupby('community_area').apply(lambda x: x.sort_values(['year','month']).tail(3)).reset_index(drop=True)


In [ ]:
predictions = []

for idx, row in future_months.iterrows():
    zone = row['community_area']
    
    # Récupérer les lags
    lags = lag_dict[zone]
    lag_3, lag_2, lag_1 = lags[-3], lags[-2], lags[-1]
    
    # Créer input
    X_row = pd.DataFrame([{
        'year': row['year'],
        'month': row['month'],
        'community_area': row['community_area'],
        'lag_1': lag_1,
        'lag_3': lag_3
    }])
    
    # Encodage zone
    X_row['community_area'] = le.transform(X_row['community_area'])
    
    # Prédire
    pred = xgb_model.predict(X_row)[0]
    
    # Stocker la prédiction
    predictions.append({
        'year': row['year'],
        'month': row['month'],
        'community_area': row['community_area'],
        'predicted_crimes': pred
    })
    
    # Mettre à jour lag_dict
    lag_dict[zone].append(pred)
    lag_dict[zone] = lag_dict[zone][-3:]  # garder seulement 3 derniers mois


In [29]:
future_predictions = pd.DataFrame(predictions)

# Si tu veux remettre le nom réel des zones (inverse LabelEncoder)
future_months['community_area'] = future_months['community_area'].astype(float)

future_predictions.head(12)


,year,month,community_area,predicted_crimes
0,2026.0,1.0,28.0,402.006287
1,2026.0,1.0,3.0,190.586227
2,2026.0,1.0,38.0,185.243484
3,2026.0,1.0,63.0,118.978745
4,2026.0,1.0,27.0,224.358521
5,2026.0,1.0,32.0,429.528290
6,2026.0,1.0,24.0,382.490326
7,2026.0,1.0,67.0,197.374176
8,2026.0,1.0,25.0,545.494751
9,2026.0,1.0,5.0,78.099236


In [30]:
agg_year_zone = future_predictions.groupby(['year','community_area'])['predicted_crimes'].sum().reset_index()


In [31]:
future_predictions['quarter'] = ((future_predictions['month']-1)//3) + 1
agg_quarter_zone = future_predictions.groupby(['year','quarter','community_area'])['predicted_crimes'].sum().reset_index()


In [32]:
future_predictions.to_csv('forecast_2026_2027.csv', index=False)
agg_year_zone.to_csv('forecast_year_zone.csv', index=False)
agg_quarter_zone.to_csv('forecast_quarter_zone.csv', index=False)
